In [ ]:
import pandas as pd
from IPython.display import display
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

from study.data import cohort, features, prediction_matrix, prediction_metadata
from study.io import WORK, load, reset_directory, write_table
from study.models import fit_model, predict
from study.observations import feature_view
from study.statistics import paired_difference, prediction_rows

In [ ]:
RULES = {'protected_variables': ['Height', 'MechVent'], 'terminal_outage_hours': 12}

In [ ]:
source_fits = load(WORK / 'models/source.pkl')
reset_directory(WORK / 'predictions/local_availability')

In [ ]:
# Collect existing full-input models for each hospital allocation
metadata, x = cohort('eicu').sort_index(), features('eicu')
shared = load(WORK / 'feature_sets.pkl')['shared']
folds = load(WORK / 'models/local.pkl')

models = {
    (repeat, fold, tier, 'full'): model
    for repeat, splits in folds.items()
    for fold, split in enumerate(splits)
    for tier, model in split['models'].items()
}

# Retrain R2 and R3 on restricted inputs
for training in ('retain_0.5', 'retain_0.25', 'known_available'):
    training_x = feature_view('eicu', x, training, 77, RULES)

    for repeat, splits in folds.items():
        for fold, split in enumerate(splits):
            for tier in ('R2', 'R3'):
                with threadpool_limits(limits = 1):
                    models[repeat, fold, tier, training] = fit_model(
                        training_x.loc[split['fit'], shared[tier]],
                        metadata.loc[split['fit'], 'in_hospital_mortality'],
                        'xgboost',
                        source_fits['parameters'],
                    )

    print('Local training input', training)

In [ ]:
# Share evaluation draws across hospital allocations and training inputs
for condition in ('full', 'retain_0.5', 'retain_0.25', 'terminal_outage', 'known_available'):
    seeds = list(range(10)) if condition.startswith('retain_') else [77]

    # Rebuild each evaluation draw before scoring the fitted models
    for draw, seed in enumerate(seeds):
        evaluated = x if condition == 'full' else feature_view('eicu', x, condition, seed, RULES)

        for repeat, splits in folds.items():
            rows = []

            for (allocation, fold, tier, training), model in models.items():
                if allocation != repeat:
                    continue

                # Compare known available inputs only with matching training inputs
                if training == 'known_available' and training != condition:
                    continue

                if condition == 'known_available' and training != condition:
                    continue

                # Score patients from the held-out hospitals
                ids = splits[fold]['test']
                rows.append(
                    prediction_rows(
                        prediction_metadata(metadata.loc[ids]),
                        predict(model, evaluated.loc[ids]),
                        repeat = repeat,
                        tier = tier,
                        training = training,
                        restriction = condition,
                        fold = fold,
                    ).assign(draw = draw)
                )

            write_table(
                WORK / 'predictions/local_availability' / f'{condition}_{draw}_{repeat}.parquet',
                pd.concat(rows, ignore_index = True),
            )

    print('Local evaluation input', condition)

In [ ]:
# Compare information increments on paired held-out patients
rows = []

for training in ('full', 'retain_0.5', 'retain_0.25', 'known_available'):
    restrictions = (
        [training] if training == 'known_available' else ['full', 'retain_0.5', 'retain_0.25', 'terminal_outage']
    )

    for restriction in restrictions:
        pairs = [('R3', 'R2')] + ([('R2', 'R1')] if training == 'full' else [])

        # Align richer and simpler tiers on the same held-out patients
        for richer, simpler in pairs:
            filters = [('training', '==', training), ('restriction', '==', restriction)]
            left = prediction_matrix('local_availability', metadata.index, filters + [('tier', '==', richer)])
            right = prediction_matrix('local_availability', metadata.index, filters + [('tier', '==', simpler)])

            for equal_hospital in (False, True):
                result = paired_difference(
                    prediction_metadata(metadata), left, right, cluster = True, equal_hospital = equal_hospital
                )

                rows.append(
                    dict(
                        training = training, restriction = restriction,
                        contrast = richer + '_minus_' + simpler, **result
                    )
                )

results = pd.DataFrame(rows)

# Adjust p values within each training and weighting family
for _, family in results.groupby(['training', 'weighting']):
    results.loc[family.index, 'holm_p'] = multipletests(family.p_value, method = 'holm')[1]

write_table(WORK / 'results/local_availability.parquet', results)

display(results.round(5))

In [ ]:
# Compare retrained R3 models under matched restrictions and full inputs
rows = []

for training in ('retain_0.5', 'retain_0.25'):
    for restriction in (training, 'full'):
        # Pair retrained and full input models under the same evaluation restriction
        filters = [('tier', '==', 'R3'), ('restriction', '==', restriction)]
        retrained = prediction_matrix(
            'local_availability', metadata.index, filters + [('training', '==', training)]
        )
        ordinary = prediction_matrix('local_availability', metadata.index, filters + [('training', '==', 'full')])

        for equal_hospital in (False, True):
            rows.append(
                dict(
                    training = training,
                    restriction = restriction,
                    **paired_difference(
                        prediction_metadata(metadata),
                        retrained,
                        ordinary,
                        cluster = True,
                        equal_hospital = equal_hospital,
                    ),
                )
            )

retraining = pd.DataFrame(rows)

# Adjust p values within each weighting family
for _, family in retraining.groupby('weighting'):
    retraining.loc[family.index, 'holm_p'] = multipletests(family.p_value, method = 'holm')[1]

write_table(WORK / 'results/local_retraining.parquet', retraining)

display(retraining.round(5))